# The same per-call lifecycle for an MCP tool

Call the server in process. The counter action and per-call declaration are the same as for HTTP.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [ ]:
!pip install -q aoa-action-machine aoa-fastapi-adapter aoa-mcp-adapter httpx

In [2]:
import asyncio
import json
from contextlib import asynccontextmanager

from pydantic import Field

from aoa.action_machine.auth import GuestRole, NoAuthCoordinator
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import BaseController, Connections
from aoa.action_machine.resources.per_call_connection import PerCallConnection
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox
from aoa.mcp import McpAdapter

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [3]:


class CounterDomain(BaseDomain):
    """Group this example's resource and action."""
    name = "counter"
    description = "Counter example"

class EmptyParams(BaseParams):
    """No input fields are needed for this case."""

@meta(description="Count calls in memory")
class CounterController(BaseController):
    """Keep a count and record lifecycle events."""
    def __init__(self, events):
        """Start with an empty count."""
        self.count = 0
        self.events = events
    @classmethod
    def get_wrapper_class(cls):
        """Allow direct use of this in-memory counter."""
        return None
    async def open(self):
        """Record opening without resetting the count."""
        self.events.append("open")
    async def close(self):
        """Record cleanup without acquiring external handles."""
        self.events.append("close")

class CountResult(BaseResult):
    """Expose the count reached by one call."""
    count: int = Field(description="Number of calls seen by this resource")

@meta(description="Increment the supplied counter", domain=CounterDomain)
@check_roles(GuestRole)
@connection(CounterController, key="counter")
class CountAction(BaseAction[EmptyParams, CountResult]):
    """Increment the supplied resource once."""
    @summary_aspect("Count one call")
    async def count_summary(self, params: EmptyParams, state: BaseState, box: ToolsBox, connections: Connections) -> CountResult:
        """Return the updated count."""
        counter = connections["counter"]
        counter.count += 1
        return CountResult(count=counter.count)


async def main():
    """Run this case and verify its observable result."""
    events = []
    @asynccontextmanager
    async def lifespan(server):
        """Surround tool calls with service startup and shutdown."""
        events.append("startup")
        try:
            yield
        finally:
            events.append("shutdown")
    server = (McpAdapter(machine=ActionProductMachine(loggers=[]), auth_coordinator=NoAuthCoordinator(context=Context()))
        .tool("count", CountAction, connections={"counter": PerCallConnection(
            lambda: CounterController(events), lambda resource: resource.close(),
        )}).build(lifespan=lifespan))
    # Direct calls do not start a network server; enter its configured lifespan explicitly.
    async with server.settings.lifespan(server):
        response = await server.call_tool("count", {})
        payload = json.loads(response.content[0].text)
        assert payload["data"] == {"count": 1}
        print(payload)
    assert events == ["startup", "open", "close", "shutdown"]
    print(events)


await main()

{'ok': True, 'code': 'OK', 'data': {'count': 1}}
['startup', 'open', 'close', 'shutdown']


## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

## 5. Explain the result

The server lifetime surrounds the call lifetime. MCP serializes the result in its own envelope; it does not change which resource is created, opened or finalized.